# New York bus data inspection

**Purpose:** understand the collected sources, test their quality and decide what
needs review before cleaning or feature engineering. This is an executed,
reproducible inspection walkthrough. Each section states a question, runs visible
checks and explains the interpretation. The reusable scripts perform batch checks;
this notebook shows the reasoning and selected examples directly.

**Scope:** inspect one historical Manhattan snapshot in depth, inspect the full
collected ridership extracts, and clearly identify the existing batch evidence
for the other GTFS versions. No models, route recommendations or crowding labels
are created here. Historical schedule comparisons are limited to the selected
Manhattan archives and their supported operating dates.

**Rerun requirements:** use the original collected files, their receipts and
denominator-query files under `data/raw/new_york`. Before running this notebook,
generate the batch evidence with `python scripts/week1_inspection.py` and the
compatibility evidence with `python scripts/validate_historical_alignment.py`.
Saved outputs can be read without rerunning these steps. Public URLs can later
return revised data, so a new download is not necessarily this frozen collection.

## 1. What does one row represent?

| Table | Meaning of one row | Key used for inspection |
|---|---|---|
| routes | A catalogue route, possibly shared across borough feeds | route_id |
| trips | A trip template activated by a service calendar | trip_id |
| stop_times | A visit to a stop within a trip | trip_id + stop_sequence |
| stops | A stop/location definition | stop_id |
| calendar | Weekly service rules with validity dates | service_id |
| calendar_dates | A dated service addition/removal | service_id + date |
| shapes | An ordered point in a route geometry | shape_id + shape_pt_sequence |
| Hourly ridership | Published boarding estimates for a route/payment clock hour | bus_route + transit_timestamp |
| Daily ridership | Published estimates summed by route/payment date | bus_route + service_date in the raw extract |

**Important:** ridership is an estimate based on electronic fares, not onboard
occupancy. Transfers are already included. Daily and hourly extracts overlap;
they are not independent passenger samples. A repeating calendar is not repeated
observation of actual buses. The daily extract's raw `service_date` alias refers
to payment dates, whereas GTFS service dates have a different midnight convention.

In [1]:
from pathlib import Path
import hashlib
import json
import zipfile

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'scripts/week1_inspection.py').exists())
RAW = ROOT / 'data/raw/new_york'
# Summary tables remain fully visible; raw-record examples use explicit head().
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print('Inputs: data/raw/new_york; source data will remain unchanged.')

Inputs: data/raw/new_york; source data will remain unchanged.


## 2. Are these the same files that were collected?

Check every receipt against its original SHA-256 and byte size. A fingerprint
verifies identity, not the correctness of the publisher's estimates. Current feed
URLs can change, so the saved bytes and capture dates matter.

In [2]:
source_rows = []
for receipt_path in sorted(RAW.rglob('*.receipt.json')):
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    path = ROOT / receipt['local_path']
    with path.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest()
    source_rows.append({
        'file': receipt['local_path'], 'bytes': path.stat().st_size,
        'hash_ok': digest == receipt['sha256'],
        'size_ok': path.stat().st_size == receipt['bytes'],
        'sha256': digest, 'retrieved_utc': receipt['retrieved_utc'],
    })
source_inventory = pd.DataFrame(source_rows)
assert not source_inventory.empty
assert source_inventory[['hash_ok', 'size_ok']].all().all()
print('Verified source files:', len(source_inventory))
print('Source size in decimal MB:', round(source_inventory.bytes.sum() / 1_000_000, 2))
display(source_inventory[['file', 'bytes', 'hash_ok', 'size_ok']])

Verified source files: 57
Source size in decimal MB: 160.59


,file,bytes,hash_ok,size_ok
0,data/raw/new_york/gtfs/archive/manhattan_20251230.zip,9647917,True,True
1,data/raw/new_york/gtfs/archive/manhattan_20260327.zip,9593304,True,True
2,data/raw/new_york/gtfs/archive/manhattan_20260409.zip,7762091,True,True
3,data/raw/new_york/gtfs/archive/manhattan_20260624.zip,6708453,True,True
4,data/raw/new_york/gtfs/archive/manhattan_20260903.zip,9519944,True,True
5,data/raw/new_york/gtfs/current/bronx.zip,8428753,True,True
6,data/raw/new_york/gtfs/current/brooklyn.zip,17046285,True,True
7,data/raw/new_york/gtfs/current/bus_company.zip,8446523,True,True
8,data/raw/new_york/gtfs/current/manhattan.zip,8322100,True,True
9,data/raw/new_york/gtfs/current/queens.zip,5983145,True,True


**Interpretation:** successful checks establish that this inspection uses the
frozen collection. The full source inventory includes data, denominator queries
and publisher documentation; it does not mean there are that many independent
datasets. Copies of source files are excluded from the inventory.

## 3. Which schedule is being inspected?

Use the 3 September 2026 Manhattan archive as the detailed example. Its archive
capture date is not its first operating date. The other versions are summarized
separately later so that a single-feed result is never presented as a network-wide test.

In [3]:
selected_zip = RAW / 'gtfs/archive/manhattan_20260903.zip'
with zipfile.ZipFile(selected_zip) as archive:
    assert archive.testzip() is None, 'ZIP member integrity failed'
    table_names = [n for n in archive.namelist() if n.endswith('.txt') and '/' not in n]
    tables = {Path(n).stem: pd.read_csv(archive.open(n), dtype='string')
              for n in table_names}
routes = tables['routes']
trips = tables['trips']
stop_times = tables['stop_times']
stops = tables['stops']
calendar = tables['calendar']
exceptions = tables['calendar_dates']
shapes = tables['shapes']
print('Selected source:', selected_zip.name)
display(tables['agency'][['agency_name', 'agency_timezone']])
display(tables['feed_info'])

Selected source: manhattan_20260903.zip


,agency_name,agency_timezone
0,MTA New York City Transit,America/New_York


,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_url
0,MTA New York City Transit,https://www.mta.info/,en,20260906,20270102,20260813,https://groups.google.com/forum/#!forum/mtadeveloperresources


## 4. What are the dimensions, schemas and memory costs?

Load entire tables for checks and show bounded previews for readability.
The next five outputs are previews only; the inspections below use all loaded rows.
GTFS columns are intentionally loaded as strings to preserve identifiers.
Numeric/date checks below use temporary parsed values; loaded types alone
do not establish that the content is valid.

In [4]:
table_overview = pd.DataFrame([
    {'table': name, 'rows': len(frame), 'columns': frame.shape[1],
     'memory_MiB': round(frame.memory_usage(deep=True).sum() / 1024**2, 2)}
    for name, frame in tables.items()
])
display(table_overview)
routes.info(memory_usage='deep')
stop_times.info(memory_usage='deep')

,table,rows,columns,memory_MiB
0,agency,1,6,0.00
1,calendar,21,10,0.01
2,calendar_dates,345,3,0.06
3,routes,307,8,0.16
4,shapes,33589,4,8.33
5,stops,1824,9,1.02
6,stop_times,1174914,8,573.54
7,trips,37872,7,17.84
8,feed_info,1,7,0.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307 entries, 0 to 306
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   route_id          307 non-null    string
 1   agency_id         307 non-null    string
 2   route_short_name  307 non-null    string
 3   route_long_name   307 non-null    string
 4   route_desc        300 non-null    string
 5   route_type        307 non-null    string
 6   route_color       307 non-null    string
 7   route_text_color  307 non-null    string
dtypes: string(8)
memory usage: 163.2 KB


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1174914 entries, 0 to 1174913
Data columns (total 8 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   trip_id         1174914 non-null  string
 1   arrival_time    1174914 non-null  string
 2   departure_time  1174914 non-null  string
 3   stop_id         1174914 non-null  string
 4   stop_sequence   1174914 non-null  string
 5   pickup_type     1174914 non-null  string
 6   drop_off_type   1174914 non-null  string
 7   timepoint       1174914 non-null  string
dtypes: string(8)
memory usage: 573.5 MB


In [5]:
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color
0,B1,MTA NYCT,B1,Bay Ridge - Manhattan Beach,via 86th St / Ocean Pkwy,3,00AEEF,FFFFFF
1,B11,MTA NYCT,B11,Sunset Park - Midwood,via 49th & 50th St / Avenue J,3,006CB7,FFFFFF
2,B12,MTA NYCT,B12,Lefferts Gardens - East New York,via Clarkson Av / Empire Blvd / East New York Av,3,6CBE45,FFFFFF
3,B13,MTA NYCT,B13,Spring Creek - Wyckoff Hospital,via Crescent St / Jamaica Av / Wyckoff Av,3,FAA61A,FFFFFF
4,B14,MTA NYCT,B14,Spring Creek - Crown Heights,via Sutter Av / Pitkin Av,3,00AEEF,FFFFFF


In [6]:
trips.head()

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id
0,M60+,MQ_O6-Weekday,MQ_O6-Weekday-009000_SBS60_901,SELECT BUS LA GUARDIA AIRPORT,0,39855329,SBS600010
1,M60+,MQ_O6-Weekday,MQ_O6-Weekday-013000_SBS60_901,SELECT BUS WEST SIDE BROADWAY-106 ST,1,39855329,SBS600011
2,M60+,MQ_O6-Weekday,MQ_O6-Weekday-019000_SBS60_901,SELECT BUS LA GUARDIA AIRPORT,0,39855329,SBS600010
3,M60+,MQ_O6-Weekday,MQ_O6-Weekday-022800_SBS60_901,SELECT BUS WEST SIDE BROADWAY-106 ST,1,39855329,SBS600011
4,M60+,MQ_O6-Weekday,MQ_O6-Weekday-029500_SBS60_906,SELECT BUS LA GUARDIA AIRPORT,0,39855337,SBS600010


In [7]:
stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,timepoint
0,MQ_O6-Weekday-009000_SBS60_901,01:30:00,01:30:00,403117,1,0,0,1
1,MQ_O6-Weekday-009000_SBS60_901,01:33:11,01:33:11,403122,2,0,0,0
2,MQ_O6-Weekday-009000_SBS60_901,01:34:31,01:34:31,404262,3,0,0,0
3,MQ_O6-Weekday-009000_SBS60_901,01:35:51,01:35:51,401442,4,0,0,0
4,MQ_O6-Weekday-009000_SBS60_901,01:37:32,01:37:32,405424,5,0,0,0


In [8]:
stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].head()

,stop_id,stop_name,stop_lat,stop_lon
0,101014,WILLIS AV/E 138 ST,40.808930,-73.922855
1,101015,WILLIS AV/E 140 ST,40.810634,-73.921632
2,101017,WILLIS AV/E 144 ST,40.813111,-73.919823
3,101018,WILLIS AV/E 146 ST,40.814404,-73.918954
4,101066,3 AV/E 150 ST,40.816142,-73.917547


In [9]:
calendar.head()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,MQ_D6-Sunday,0,0,0,0,0,0,1,20260906,20270101
1,MQ_D6-Weekday-SDon,1,1,1,1,1,0,0,20260908,20261223
2,MQ_D6-Saturday,0,0,0,0,0,1,0,20260912,20270102
3,MQ_O6-Weekday,1,1,1,1,1,0,0,20261012,20261231
4,MV_D6-Sunday,0,0,0,0,0,0,1,20260906,20270101


**Interpretation:** route catalogue rows and trip templates describe different
things. `route_id`, `trip_id`, `stop_id`, `service_id` and `shape_id` are identifiers,
even where their values resemble numbers. Deep memory includes string content.

## 5. Which values are missing, blank or padded with spaces?

Report missingness for every column. Separately inspect the fields needed for
this analysis. These project requirements are not a complete GTFS specification:
for example, some intermediate stop times are conditionally optional in GTFS.
Missing optional descriptions are not automatically data errors.

In [10]:
missing_profile = pd.concat([
    pd.DataFrame({'table': name, 'column': frame.columns,
                  'missing_count': frame.isna().sum().to_numpy(),
                  'missing_percent': frame.isna().mean().mul(100).round(2).to_numpy()})
    for name, frame in tables.items()
], ignore_index=True)
print('Missing-value results for every inspected column, including zero counts:')
display(missing_profile)
print('Columns with missing values:')
display(missing_profile[missing_profile.missing_count.gt(0)])

Missing-value results for every inspected column, including zero counts:


,table,column,missing_count,missing_percent
0,agency,agency_id,0,0.00
1,agency,agency_name,0,0.00
2,agency,agency_url,0,0.00
3,agency,agency_timezone,0,0.00
4,agency,agency_lang,0,0.00
5,agency,agency_phone,0,0.00
6,calendar,service_id,0,0.00
7,calendar,monday,0,0.00
8,calendar,tuesday,0,0.00
9,calendar,wednesday,0,0.00


Columns with missing values:


,table,column,missing_count,missing_percent
23,routes,route_desc,7,2.28
33,stops,stop_desc,1824,100.00
36,stops,zone_id,1824,100.00
37,stops,stop_url,1824,100.00
39,stops,parent_station,1824,100.00


In [11]:
analysis_fields = {
    'routes': ['route_id', 'agency_id', 'route_short_name', 'route_type'],
    'trips': ['trip_id', 'route_id', 'service_id', 'shape_id'],
    'stop_times': ['trip_id', 'stop_id', 'stop_sequence', 'arrival_time', 'departure_time'],
    'stops': ['stop_id', 'stop_name', 'stop_lat', 'stop_lon'],
    'calendar': ['service_id', 'start_date', 'end_date'],
    'shapes': ['shape_id', 'shape_pt_sequence', 'shape_pt_lat', 'shape_pt_lon'],
}
field_checks = []
for name, fields in analysis_fields.items():
    for field in fields:
        values = tables[name][field]
        stripped = values.str.strip()
        field_checks.append({
            'table': name, 'field': field, 'missing': int(values.isna().sum()),
            'blank_nonmissing': int(stripped.eq('').sum()),
            'outer_spaces': int(values.ne(stripped).sum()),
        })
field_checks = pd.DataFrame(field_checks)
display(field_checks)
print('Totals:', field_checks[['missing', 'blank_nonmissing', 'outer_spaces']].sum().to_dict())
padded_fields = field_checks.loc[field_checks.outer_spaces.gt(0)]
display(padded_fields)
padding_examples = []
for row in padded_fields.itertuples(index=False):
    values = tables[row.table][row.field]
    for value in values.loc[values.ne(values.str.strip())].head(2):
        padding_examples.append({'table': row.table, 'field': row.field,
                                 'original_value_repr': repr(value),
                                 'stripped_value_repr': repr(value.strip())})
display(pd.DataFrame(padding_examples))

,table,field,missing,blank_nonmissing,outer_spaces
0,routes,route_id,0,0,0
1,routes,agency_id,0,0,0
2,routes,route_short_name,0,0,0
3,routes,route_type,0,0,0
4,trips,trip_id,0,0,0
5,trips,route_id,0,0,0
6,trips,service_id,0,0,0
7,trips,shape_id,0,0,0
8,stop_times,trip_id,0,0,0
9,stop_times,stop_id,0,0,0


Totals: {'missing': 0, 'blank_nonmissing': 0, 'outer_spaces': 3648}


,table,field,missing,blank_nonmissing,outer_spaces
15,stops,stop_lat,0,0,1824
16,stops,stop_lon,0,0,1824


,table,field,original_value_repr,stripped_value_repr
0,stops,stop_lat,' 40.808930','40.808930'
1,stops,stop_lat,' 40.810634','40.810634'
2,stops,stop_lon,' -73.922855','-73.922855'
3,stops,stop_lon,' -73.921632','-73.921632'


**Decision:** retain optional nulls and original strings. A nonzero count in a
required analysis field needs investigation before joining or measuring service.
Do not silently strip identifiers or impute values during inspection.
The examples distinguish padding in numeric coordinate text from padding in
identifiers. Numeric text can be explicitly parsed in a temporary diagnostic;
this does not justify changing route/stop IDs or overwriting the raw source.

## 6. Are rows and identifiers unique at the correct grain?

A repeated route name is not necessarily a duplicate route. A repeated stop
within a trip may be a valid revisit. Test declared keys rather than deleting
rows because their labels look similar.

In [12]:
key_definitions = {
    'routes': ['route_id'], 'trips': ['trip_id'], 'stops': ['stop_id'],
    'stop_times': ['trip_id', 'stop_sequence'], 'calendar': ['service_id'],
    'calendar_dates': ['service_id', 'date'],
    'shapes': ['shape_id', 'shape_pt_sequence'],
}
key_checks = pd.DataFrame([
    {'table': name, 'key': ' + '.join(keys),
     'duplicate_key_rows': int(tables[name].duplicated(keys).sum()),
     'duplicate_full_rows': int(tables[name].duplicated().sum())}
    for name, keys in key_definitions.items()
])
display(key_checks)

,table,key,duplicate_key_rows,duplicate_full_rows
0,routes,route_id,0,0
1,trips,trip_id,0,0
2,stops,stop_id,0,0
3,stop_times,trip_id + stop_sequence,0,0
4,calendar,service_id,0,0
5,calendar_dates,service_id + date,0,0
6,shapes,shape_id + shape_pt_sequence,0,0


In [13]:
repeated_names = routes.loc[routes.route_long_name.duplicated(keep=False),
                            ['route_id', 'route_short_name', 'route_long_name']]
print('Catalogue rows sharing a long name:', len(repeated_names))
display(repeated_names.head(8))
visits_per_trip = stop_times.groupby('trip_id').agg(
    stop_visits=('stop_id', 'size'), distinct_stops=('stop_id', 'nunique'))
print('Trips revisiting a stop:', int(visits_per_trip.stop_visits.gt(visits_per_trip.distinct_stops).sum()))
print('Single-stop trip templates:', int(visits_per_trip.stop_visits.eq(1).sum()))
display(visits_per_trip.describe().round(2))

Catalogue rows sharing a long name: 72


,route_id,route_short_name,route_long_name
12,B26,B26,Downtown Brooklyn - Ridgewood
25,B44,B44,Sheepshead Bay - Williamsburg
26,B44+,B44-SBS,Sheepshead Bay - Williamsburg
27,B45,B45,Downtown Brooklyn - Crown Heights
28,B46,B46,Kings Plaza - Williamsburg
29,B46+,B46-SBS,Kings Plaza - Williamsburg
33,B52,B52,Downtown Brooklyn - Ridgewood
34,B54,B54,Downtown Brooklyn - Ridgewood


Trips revisiting a stop: 0
Single-stop trip templates: 0


,stop_visits,distinct_stops
count,37872.00,37872.00
mean,31.02,31.02
std,19.67,19.67
min,5.00,5.00
25%,16.00,16.00
50%,21.00,21.00
75%,47.00,47.00
max,81.00,81.00


## 7. Do references connect to existing records?

Broken references prevent a reliable route-to-trip-to-stop analysis. Conversely,
a catalogue route without trips in this ZIP can simply belong to another borough.

In [14]:
known_services = set(calendar.service_id) | set(exceptions.service_id)
references = pd.DataFrame([
    ['trips -> routes', int((~trips.route_id.isin(routes.route_id)).sum())],
    ['trips -> service calendar', int((~trips.service_id.isin(known_services)).sum())],
    ['stop_times -> trips', int((~stop_times.trip_id.isin(trips.trip_id)).sum())],
    ['stop_times -> stops', int((~stop_times.stop_id.isin(stops.stop_id)).sum())],
    ['trips -> shapes', int((~trips.shape_id.isin(shapes.shape_id)).sum())],
    ['trips without stop visits', int((~trips.trip_id.isin(stop_times.trip_id)).sum())],
], columns=['relationship', 'unmatched_rows'])
display(references)
print('Catalogue routes:', len(routes))
print('Routes actually referenced by trip templates:', trips.route_id.nunique())
catalogue_only = routes.loc[~routes.route_id.isin(trips.route_id)]
print('Catalogue-only route rows:', len(catalogue_only))
display(catalogue_only[['route_id', 'route_short_name', 'route_long_name']].head())

,relationship,unmatched_rows
0,trips -> routes,0
1,trips -> service calendar,0
2,stop_times -> trips,0
3,stop_times -> stops,0
4,trips -> shapes,0
5,trips without stop visits,0


Catalogue routes: 307
Routes actually referenced by trip templates: 41
Catalogue-only route rows: 266


,route_id,route_short_name,route_long_name
0,B1,B1,Bay Ridge - Manhattan Beach
1,B11,B11,Sunset Park - Midwood
2,B12,B12,Lefferts Gardens - East New York
3,B13,B13,Spring Creek - Wyckoff Hospital
4,B14,B14,Spring Creek - Crown Heights


**Interpretation:** the Manhattan file contains a shared NYCT catalogue. Use trip
references when describing its scheduled route scope, and do not sum the route
catalogue across versions or boroughs as if all rows were distinct services.

## 8. Are numeric fields and category values plausible?

Check parsed coordinates, sequence numbers and selected category domains.
Worldwide coordinate validity is a basic check; it does not prove a stop is
correctly placed on a road or its route geometry.

In [15]:
stop_coordinates = stops[['stop_lat', 'stop_lon']].apply(pd.to_numeric, errors='coerce')
shape_coordinates = shapes[['shape_pt_lat', 'shape_pt_lon']].apply(pd.to_numeric, errors='coerce')
coordinate_checks = pd.DataFrame([
    ['stop_lat', int((~stop_coordinates.stop_lat.between(-90, 90)).sum())],
    ['stop_lon', int((~stop_coordinates.stop_lon.between(-180, 180)).sum())],
    ['shape_pt_lat', int((~shape_coordinates.shape_pt_lat.between(-90, 90)).sum())],
    ['shape_pt_lon', int((~shape_coordinates.shape_pt_lon.between(-180, 180)).sum())],
], columns=['field', 'invalid_or_missing'])
display(coordinate_checks)
display(stop_coordinates.describe().round(6))
extreme_indices = {stop_coordinates.stop_lat.idxmin(), stop_coordinates.stop_lat.idxmax(),
                   stop_coordinates.stop_lon.idxmin(), stop_coordinates.stop_lon.idxmax()}
display(stops.loc[sorted(extreme_indices), ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']])

,field,invalid_or_missing
0,stop_lat,0
1,stop_lon,0
2,shape_pt_lat,0
3,shape_pt_lon,0


,stop_lat,stop_lon
count,1824.0,1824.0
mean,40.774052,-73.968605
std,0.037678,0.024212
min,40.701556,-74.018088
25%,40.744318,-73.986792
50%,40.772986,-73.970099
75%,40.803182,-73.949155
max,40.865166,-73.864443


,stop_id,stop_name,stop_lat,stop_lon
1479,404980,BATTERY PL/1 PL,40.705837,-74.018088
1515,405083,SOUTH ST/WHITEHALL ST,40.701556,-74.012461
1738,503848,LAGUARDIA AIRPORT/TERMINAL C,40.770328,-73.864443
1759,803067,MARGARET CORBIN DR/CLOISTERS,40.865166,-73.930841


In [16]:
sequence = pd.to_numeric(stop_times.stop_sequence, errors='coerce')
shape_sequence = pd.to_numeric(shapes.shape_pt_sequence, errors='coerce')
category_checks = pd.DataFrame([
    ['direction_id', int((~trips.direction_id.dropna().isin(['0', '1'])).sum())],
    ['calendar exception_type', int((~exceptions.exception_type.isin(['1', '2'])).sum())],
    ['stop_sequence', int((sequence.isna() | sequence.lt(0) | sequence.mod(1).ne(0)).sum())],
    ['shape_pt_sequence', int((shape_sequence.isna() | shape_sequence.lt(0) | shape_sequence.mod(1).ne(0)).sum())],
], columns=['field', 'invalid_rows'])
display(category_checks)
print('Catalogue route types:', routes.route_type.value_counts(dropna=False).to_dict())
print('Trip directions:', trips.direction_id.value_counts(dropna=False).to_dict())

,field,invalid_rows
0,direction_id,0
1,calendar exception_type,0
2,stop_sequence,0
3,shape_pt_sequence,0


Catalogue route types: {'3': 255, '711': 52}
Trip directions: {'1': 18961, '0': 18911}


## 9. What calendar dates can this archive actually represent?

Check parsed bounds, weekday flags and dated exceptions. A service ID's name
does not establish its operating days. Feed validity and service calendars are
both relevant, especially at a schedule changeover.

In [17]:
day_columns = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
start_dates = pd.to_datetime(calendar.start_date, format='%Y%m%d', errors='coerce')
end_dates = pd.to_datetime(calendar.end_date, format='%Y%m%d', errors='coerce')
exception_dates = pd.to_datetime(exceptions.date, format='%Y%m%d', errors='coerce')
calendar_checks = pd.Series({
    'invalid_start_dates': int(start_dates.isna().sum()),
    'invalid_end_dates': int(end_dates.isna().sum()),
    'reversed_date_ranges': int(start_dates.gt(end_dates).sum()),
    'invalid_weekday_rows': int((~calendar[day_columns].isin(['0', '1'])).any(axis=1).sum()),
    'no_active_weekday_rows': int(calendar[day_columns].eq('0').all(axis=1).sum()),
    'invalid_exception_dates': int(exception_dates.isna().sum()),
})
display(calendar_checks.to_frame('count'))
print('Weekly calendar bounds:', start_dates.min().date(), 'to', end_dates.max().date())
print('Dated additions/removals:', exceptions.exception_type.value_counts().to_dict())
display(exceptions.head(8))

,count
invalid_start_dates,0
invalid_end_dates,0
reversed_date_ranges,0
invalid_weekday_rows,0
no_active_weekday_rows,0
invalid_exception_dates,0


Weekly calendar bounds: 2026-09-05 to 2027-01-02
Dated additions/removals: {'2': 300, '1': 45}


,service_id,date,exception_type
0,MQ_D6-Sunday,20260907,1
1,MQ_D6-Sunday,20261126,1
2,MQ_D6-Sunday,20261225,1
3,MQ_D6-Sunday,20270101,1
4,MQ_D6-Weekday-SDon,20261012,2
5,MQ_D6-Weekday-SDon,20261126,2
6,MQ_D6-Weekday-SDon,20261127,2
7,MQ_D6-Weekday-SDon,20260908,2


In [18]:
calendar_examples = []
for date in pd.to_datetime(['2026-09-05', '2026-09-07', '2026-09-08']):
    key = date.strftime('%Y%m%d')
    weekly_mask = (start_dates.le(date) & end_dates.ge(date)
                   & calendar[day_columns[date.weekday()]].eq('1'))
    weekly = set(calendar.loc[weekly_mask, 'service_id'])
    changes = exceptions.loc[exceptions.date.eq(key)]
    removed = set(changes.loc[changes.exception_type.eq('2'), 'service_id'])
    added = set(changes.loc[changes.exception_type.eq('1'), 'service_id'])
    active = (weekly - removed) | added
    info = tables['feed_info'].iloc[0]
    within_feed = info.feed_start_date <= key <= info.feed_end_date
    calendar_examples.append({
        'date': date.date(), 'weekday': date.day_name(),
        'weekly_services': len(weekly), 'removed': len(removed), 'added': len(added),
        'within_feed_info': within_feed,
        'eligible_templates': int(trips.service_id.isin(active).sum()) if within_feed else None,
    })
display(pd.DataFrame(calendar_examples))

,date,weekday,weekly_services,removed,added,within_feed_info,eligible_templates
0,2026-09-05,Saturday,1,0,0,False,NaN
1,2026-09-07,Monday,1,1,5,True,6231.0
2,2026-09-08,Tuesday,5,5,5,True,8569.0


**Decision:** exclude dates outside this snapshot's stated validity. The holiday
example demonstrates additions/removals; it does not prove that scheduled trips
actually operated. The separate historical audit selects one whole archive per
date before applying calendars and never silently falls back to an older route.

## 10. Are times parseable and ordered within each trip?

GTFS permits hours beyond 23. Keep them as elapsed service-day seconds for checks;
using a normal clock parser or modulo 24 would lose the day offset. These are
temporary diagnostics, not a saved duration/frequency feature table.

In [19]:
time_format = r'\d{2,}:[0-5]\d:[0-5]\d'
time_checks = []
for field in ['arrival_time', 'departure_time']:
    time_checks.append({'field': field,
                        'invalid_or_missing': int((~stop_times[field].str.fullmatch(time_format, na=False)).sum())})
display(pd.DataFrame(time_checks))
timing = stop_times[['trip_id', 'stop_id']].copy()
timing['stop_sequence'] = sequence
timing['arrival_seconds'] = pd.to_timedelta(stop_times.arrival_time, errors='coerce').dt.total_seconds()
timing['departure_seconds'] = pd.to_timedelta(stop_times.departure_time, errors='coerce').dt.total_seconds()
timing = timing.sort_values(['trip_id', 'stop_sequence'])
timing['previous_departure'] = timing.groupby('trip_id').departure_seconds.shift()
timing['travel_gap_minutes'] = (timing.arrival_seconds - timing.previous_departure) / 60
timing['dwell_minutes'] = (timing.departure_seconds - timing.arrival_seconds) / 60
print('Departure before arrival:', int(timing.departure_seconds.lt(timing.arrival_seconds).sum()))
print('Arrival before preceding departure:', int(timing.arrival_seconds.lt(timing.previous_departure).sum()))
extended = stop_times.loc[timing.index[timing.departure_seconds.ge(86400)],
                          ['trip_id', 'stop_sequence', 'arrival_time', 'departure_time']]
print('Stop visits departing at hour 24 or later:', len(extended))
display(extended.head())

,field,invalid_or_missing
0,arrival_time,0
1,departure_time,0


Departure before arrival: 0
Arrival before preceding departure: 0


Stop visits departing at hour 24 or later: 41450


,trip_id,stop_sequence,arrival_time,departure_time
310201,MQ_D6-Saturday-140000_SBS60_938,12,24:00:34,24:00:34
310202,MQ_D6-Saturday-140000_SBS60_938,13,24:01:57,24:01:57
310203,MQ_D6-Saturday-140000_SBS60_938,14,24:03:10,24:03:10
310204,MQ_D6-Saturday-140000_SBS60_938,15,24:04:11,24:04:11
310205,MQ_D6-Saturday-140000_SBS60_938,16,24:07:00,24:07:00


In [20]:
trip_timing = timing.groupby('trip_id').agg(
    first_departure=('departure_seconds', 'first'), final_arrival=('arrival_seconds', 'last'))
inspection_duration = (trip_timing.final_arrival - trip_timing.first_departure) / 60
print('Negative trip durations:', int(inspection_duration.lt(0).sum()))
print('Zero trip durations:', int(inspection_duration.eq(0).sum()))
print('Trip durations over 300 minutes (review threshold):', int(inspection_duration.gt(300).sum()))
display(inspection_duration.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_frame('scheduled_minutes'))
print('Stop visits with travel gaps over 30 minutes:', int(timing.travel_gap_minutes.gt(30).sum()))
print('Stop visits with dwell over 30 minutes:', int(timing.dwell_minutes.gt(30).sum()))
display(timing.nlargest(5, 'travel_gap_minutes')[
    ['trip_id', 'stop_id', 'stop_sequence', 'travel_gap_minutes', 'dwell_minutes']])

Negative trip durations: 0
Zero trip durations: 0
Trip durations over 300 minutes (review threshold): 0


,scheduled_minutes
count,37872.000000
mean,45.509559
std,27.690053
min,5.000000
1%,12.000000
25%,24.000000
50%,35.000000
75%,62.000000
99%,124.000000
max,139.000000


Stop visits with travel gaps over 30 minutes: 0
Stop visits with dwell over 30 minutes: 0


,trip_id,stop_id,stop_sequence,travel_gap_minutes,dwell_minutes
429943,MV_D6-Weekday-093500_M98_903,403439,9,26.416667,0.0
429961,MV_D6-Weekday-095000_M98_905,403439,9,26.416667,0.0
429979,MV_D6-Weekday-096500_M98_910,403439,9,26.416667,0.0
429997,MV_D6-Weekday-098000_M98_907,403439,9,26.416667,0.0
430015,MV_D6-Weekday-099500_M98_911,403439,9,26.416667,0.0


**Interpretation:** a nonnegative duration can still be implausible. Large gaps
are screening examples, not measured delays or automatic errors. Overnight trips
are legitimate but require a separate payment-date/service-date compatibility check.

## 11. What is checked for the other ten GTFS snapshots?

The full batch inspection already loads their tables and saves per-column,
key and reference checks. Show that saved evidence explicitly, including its
timestamp. This cell displays saved results. The fingerprint checks above
verify that the raw versions match the original collection.

In [21]:
prior_summary = json.loads((ROOT / 'reports/week1/inspection_summary.json').read_text())
previous_inventory = pd.read_csv(ROOT / 'reports/week1/source_inventory.csv')
assert dict(zip(previous_inventory.local_path, previous_inventory.sha256)) == dict(
    zip(source_inventory.file, source_inventory.sha256)), 'Batch evidence uses different inputs'
print('Saved batch inspection UTC:', prior_summary['inspected_utc'])
feed_overview = pd.read_csv(ROOT / 'reports/week1/gtfs_inventory.csv')
display(feed_overview[['feed', 'routes', 'trips', 'stop_times', 'stops',
                       'routes_referenced_by_trips', 'calendar_start', 'calendar_end']])
all_batch_checks = []
for check_file in sorted((ROOT / 'reports/week1/gtfs').glob('*/key_and_field_checks.csv')):
    check = pd.read_csv(check_file)
    check.insert(0, 'feed', check_file.parent.name)
    all_batch_checks.append(check)
all_batch_checks = pd.concat(all_batch_checks, ignore_index=True)
print('All saved checks by feed; zero means checked with no flagged rows, NaN means not recorded:')
display(all_batch_checks.pivot(index='check', columns='feed', values='count'))
batch_flags = all_batch_checks.loc[all_batch_checks['count'].gt(0)]
print('Nonzero saved checks requiring context or review:')
display(batch_flags[['feed', 'check', 'count']])

Saved batch inspection UTC: 2026-10-27T07:21:21.985012+00:00


,feed,routes,trips,stop_times,stops,routes_referenced_by_trips,calendar_start,calendar_end
0,archive_manhattan_20251230,294,37864,1177546,1828,41,2026-01-03,2026-03-28
1,archive_manhattan_20260327,294,37864,1177546,1828,41,2026-01-03,2026-04-11
2,archive_manhattan_20260409,294,30289,947151,1830,41,2026-04-11,2026-06-27
3,archive_manhattan_20260624,307,30285,946079,1825,41,2026-06-27,2026-09-05
4,archive_manhattan_20260903,307,37872,1174914,1824,41,2026-09-05,2027-01-02
5,current_bronx,307,41156,1160917,1890,46,2026-10-08,2027-01-02
6,current_brooklyn,307,58357,2399470,4575,68,2026-10-08,2027-01-02
7,current_bus_company,92,45424,1077998,2758,92,2026-10-09,2027-01-02
8,current_manhattan,307,37872,1174640,1823,41,2026-10-08,2027-01-02
9,current_queens,307,31199,817866,1422,40,2026-10-08,2027-01-02


All saved checks by feed; zero means checked with no flagged rows, NaN means not recorded:


feed,archive_manhattan_20251230,archive_manhattan_20260327,archive_manhattan_20260409,archive_manhattan_20260624,archive_manhattan_20260903,current_bronx,current_brooklyn,current_bus_company,current_manhattan,current_queens,current_staten_island
check,,,,,,,,,,,
calendar_duplicate_keys,0,0,0,0,0,0,0,0,0,0,0
catalogue_routes_without_trip_templates,253,253,253,266,266,261,239,0,266,267,249
duplicate_service_date_exceptions,0,0,0,0,0,0,0,0,0,0,0
duplicate_shape_sequence_keys,0,0,0,0,0,0,0,0,0,0,0
invalid_calendar_date_rows,0,0,0,0,0,0,0,0,0,0,0
invalid_exception_dates,0,0,0,0,0,0,0,0,0,0,0
invalid_exception_types,0,0,0,0,0,0,0,0,0,0,0
invalid_or_missing_stop_lat,0,0,0,0,0,0,0,0,0,0,0
invalid_or_missing_stop_lon,0,0,0,0,0,0,0,0,0,0,0


Nonzero saved checks requiring context or review:


,feed,check,count
13,archive_manhattan_20251230,catalogue_routes_without_trip_templates,253
35,archive_manhattan_20260327,catalogue_routes_without_trip_templates,253
57,archive_manhattan_20260409,catalogue_routes_without_trip_templates,253
79,archive_manhattan_20260624,catalogue_routes_without_trip_templates,266
101,archive_manhattan_20260903,catalogue_routes_without_trip_templates,266
123,current_bronx,catalogue_routes_without_trip_templates,261
145,current_brooklyn,catalogue_routes_without_trip_templates,239
189,current_manhattan,catalogue_routes_without_trip_templates,266
211,current_queens,catalogue_routes_without_trip_templates,267
233,current_staten_island,catalogue_routes_without_trip_templates,249


**Interpretation:** the displayed catalogue-only rows require context, not deletion.
Selected basic checks passing is not comprehensive GTFS certification. Detailed
time-order checks above concern the chosen archive; the historical alignment
script separately checks all five archived feeds, not all current borough feeds.

## 12. What passenger-demand data is available?

Load the full collected extracts directly. Daily history and the hourly pilot
have different windows and must stay separate. Keep publisher aggregates intact.

In [22]:
daily_files = sorted((RAW / 'history_daily').glob('*_route_day.csv'))
hourly_files = sorted((RAW / 'ridership').glob('*_route_hour.csv'))
daily = pd.concat([pd.read_csv(p, dtype={'bus_route': 'string'}) for p in daily_files], ignore_index=True)
hourly = pd.concat([pd.read_csv(p, dtype={'bus_route': 'string'}) for p in hourly_files], ignore_index=True)
daily_dates = pd.to_datetime(daily.service_date, errors='coerce').dt.normalize()
hourly_times = pd.to_datetime(hourly.transit_timestamp, errors='coerce')
display(pd.DataFrame([
    ['Daily payment-date aggregate', len(daily_files), len(daily), daily.shape[1], daily_dates.min(), daily_dates.max()],
    ['Hourly payment-clock aggregate', len(hourly_files), len(hourly), hourly.shape[1], hourly_times.min(), hourly_times.max()],
], columns=['extract', 'files', 'rows', 'columns', 'first_timestamp', 'last_timestamp']))
display(daily.head())
display(hourly.head())

,extract,files,rows,columns,first_timestamp,last_timestamp
0,Daily payment-date aggregate,7,275896,6,2025-01-01,2026-09-30 00:00:00
1,Hourly payment-clock aggregate,14,993600,5,2026-07-01,2026-09-30 23:00:00


,service_date,bus_route,published_boardings,published_transfers,reported_hours,source_rows
0,2025-01-01T00:00:00.000,B1,5246,864,24,576
1,2025-01-01T00:00:00.000,B100,711,185,24,576
2,2025-01-01T00:00:00.000,B101,0,0,24,576
3,2025-01-01T00:00:00.000,B103,2423,438,24,576
4,2025-01-01T00:00:00.000,B106,0,0,24,576


,transit_timestamp,bus_route,ridership,transfers,source_rows
0,2026-07-01T00:00:00.000,B1,26,6,26
1,2026-07-01T00:00:00.000,B100,8,5,26
2,2026-07-01T00:00:00.000,B101,0,0,26
3,2026-07-01T00:00:00.000,B102,0,0,26
4,2026-07-01T00:00:00.000,B103,31,15,26


In [23]:
daily.info(memory_usage='deep')
hourly.info(memory_usage='deep')
demand_missing = pd.concat([
    pd.DataFrame({'extract': name, 'column': frame.columns,
                  'loaded_dtype': frame.dtypes.astype(str).to_numpy(),
                  'missing_count': frame.isna().sum().to_numpy(),
                  'missing_percent': frame.isna().mean().mul(100).round(2).to_numpy()})
    for name, frame in [('daily', daily), ('hourly', hourly)]
], ignore_index=True)
display(demand_missing)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275896 entries, 0 to 275895
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   service_date         275896 non-null  object
 1   bus_route            275896 non-null  string
 2   published_boardings  275896 non-null  int64 
 3   published_transfers  275896 non-null  int64 
 4   reported_hours       275896 non-null  int64 
 5   source_rows          275896 non-null  int64 
dtypes: int64(4), object(1), string(1)
memory usage: 45.4 MB


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 993600 entries, 0 to 993599
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   transit_timestamp  993600 non-null  object
 1   bus_route          993600 non-null  string
 2   ridership          993600 non-null  int64 
 3   transfers          993600 non-null  int64 
 4   source_rows        993600 non-null  int64 
dtypes: int64(3), object(1), string(1)
memory usage: 155.8 MB


,extract,column,loaded_dtype,missing_count,missing_percent
0,daily,service_date,object,0,0.0
1,daily,bus_route,string,0,0.0
2,daily,published_boardings,int64,0,0.0
3,daily,published_transfers,int64,0,0.0
4,daily,reported_hours,int64,0,0.0
5,daily,source_rows,int64,0,0.0
6,hourly,transit_timestamp,object,0,0.0
7,hourly,bus_route,string,0,0.0
8,hourly,ridership,int64,0,0.0
9,hourly,transfers,int64,0,0.0


## 13. Are ridership keys and numeric values valid?

Test missing values, route formatting, duplicate route/time keys, negative or
fractional values and whether transfers exceed total boardings. Never replace
invalid values with zero just to make these checks pass.

In [24]:
demand_checks = []
for name, frame, date_values, time_column, boardings in [
    ('daily', daily, daily_dates, 'service_date', 'published_boardings'),
    ('hourly', hourly, hourly_times, 'transit_timestamp', 'ridership'),
]:
    transfer = 'published_transfers' if name == 'daily' else 'transfers'
    demand_checks.append({
        'extract': name, 'invalid_dates': int(date_values.isna().sum()),
        'duplicate_keys': int(frame.duplicated(['bus_route', time_column]).sum()),
        'missing_routes': int(frame.bus_route.isna().sum()),
        'blank_routes': int(frame.bus_route.str.strip().eq('').sum()),
        'padded_routes': int(frame.bus_route.ne(frame.bus_route.str.strip()).sum()),
        'transfers_exceed_boardings': int(frame[transfer].gt(frame[boardings]).sum()),
    })
    for field in [boardings, transfer, 'source_rows']:
        values = pd.to_numeric(frame[field], errors='coerce')
        print(name, field, {
            'missing_or_invalid': int(values.isna().sum()),
            'negative': int(values.lt(0).sum()), 'fractional': int(values.dropna().mod(1).ne(0).sum())})
demand_checks = pd.DataFrame(demand_checks)
display(demand_checks)
print('Hourly timestamps not on an exact hour:', int(
    (hourly_times.dt.minute.ne(0) | hourly_times.dt.second.ne(0)).sum()))

daily published_boardings {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}
daily published_transfers {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}


daily source_rows {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}


hourly ridership {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}
hourly transfers {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}
hourly source_rows {'missing_or_invalid': 0, 'negative': 0, 'fractional': 0}


,extract,invalid_dates,duplicate_keys,missing_routes,blank_routes,padded_routes,transfers_exceed_boardings
0,daily,0,0,0,0,0,0
1,hourly,0,0,0,0,0,0


Hourly timestamps not on an exact hour: 0


In [25]:
display(daily[['published_boardings', 'published_transfers', 'reported_hours']].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2))
display(hourly[['ridership', 'transfers', 'source_rows']].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2))
print('Published zero boarding rows:', {
    'daily': int(daily.published_boardings.eq(0).sum()),
    'hourly': int(hourly.ridership.eq(0).sum()),
})
display(daily.nlargest(5, 'published_boardings')[['service_date', 'bus_route', 'published_boardings', 'reported_hours']])

,published_boardings,published_transfers,reported_hours
count,275896.00,275896.00,275896.0
mean,2505.81,594.00,24.0
std,3219.28,804.43,0.0
min,0.00,0.00,24.0
1%,0.00,0.00,24.0
25%,0.00,0.00,24.0
50%,1291.00,249.00,24.0
75%,3843.00,914.00,24.0
99%,14188.05,3616.00,24.0
max,26846.00,6101.00,24.0


,ridership,transfers,source_rows
count,993600.00,993600.00,993600.0
mean,97.42,25.80,26.0
std,172.04,49.32,0.0
min,0.00,0.00,26.0
1%,0.00,0.00,26.0
25%,0.00,0.00,26.0
50%,18.00,3.00,26.0
75%,124.00,30.00,26.0
99%,778.00,232.00,26.0
max,3456.00,835.00,26.0


Published zero boarding rows: {'daily': 71125, 'hourly': 398896}


,service_date,bus_route,published_boardings,reported_hours
275610,2026-09-30T00:00:00.000,M15+,26846,24
225660,2026-06-11T00:00:00.000,M15+,26825,24
270210,2026-09-18T00:00:00.000,M15+,26311,24
269760,2026-09-17T00:00:00.000,M15+,26292,24
203154,2026-04-21T00:00:00.000,M15+,26121,24


**Interpretation:** a high value is not automatically an outlier to remove; it may
describe a busy route. A zero is an explicit source value, while an absent row is
different. Neither establishes the number of people onboard a vehicle.

## 14. Are entire route/dates or hours absent despite no missing cells?

Cell-level null checks cannot detect missing records. Count coverage within each
route's observed first-to-last window and across the full extract window. A missing
route/date could reflect a retired or new code, missing publication or no service;
it is not automatically a collection failure.

In [26]:
daily_key_dates = pd.DataFrame({'bus_route': daily.bus_route, 'payment_date': daily_dates})
coverage = daily_key_dates.groupby('bus_route').payment_date.agg(
    first_date='min', last_date='max', observed_dates='nunique')
coverage['dates_in_observed_span'] = (coverage.last_date - coverage.first_date).dt.days + 1
coverage['absent_dates_inside_span'] = coverage.dates_in_observed_span - coverage.observed_dates
full_window_days = (daily_dates.max() - daily_dates.min()).days + 1
coverage['absent_dates_in_full_window'] = full_window_days - coverage.observed_dates
print('Daily routes:', len(coverage), '| Full window dates:', full_window_days)
print('Absent route/date pairs within observed spans:', int(coverage.absent_dates_inside_span.sum()))
print('Absent route/date pairs across the entire extract window:', int(coverage.absent_dates_in_full_window.sum()))
display(coverage.sort_values('absent_dates_inside_span', ascending=False).head(10))

Daily routes: 450 | Full window dates: 638
Absent route/date pairs within observed spans: 0
Absent route/date pairs across the entire extract window: 11204


,first_date,last_date,observed_dates,dates_in_observed_span,absent_dates_inside_span,absent_dates_in_full_window
bus_route,,,,,,
X9,2025-01-01,2026-09-30,638,638,0,0
B1,2025-01-01,2026-09-30,638,638,0,0
X22A,2025-01-01,2026-09-30,638,638,0,0
X22,2025-01-01,2026-09-30,638,638,0,0
X21,2025-01-01,2026-09-30,638,638,0,0
X2,2025-01-01,2026-09-30,638,638,0,0
X19,2025-01-01,2026-09-30,638,638,0,0
X17J,2025-01-01,2026-09-30,638,638,0,0
X17A,2025-01-01,2026-09-30,638,638,0,0


In [27]:
# July-September 2026 has no local daylight-saving transition, so 24 clock hours apply.
hourly_coverage = hourly.assign(payment_date=hourly_times.dt.normalize()).groupby(
    ['bus_route', 'payment_date']).agg(reported_hours=('transit_timestamp', 'nunique'))
partial_route_days = hourly_coverage.loc[hourly_coverage.reported_hours.ne(24)]
expected_hourly_rows = hourly.bus_route.nunique() * hourly_times.dt.normalize().nunique() * 24
print('Hourly rows:', len(hourly), '| Full route/date/hour grid:', expected_hourly_rows)
print('Observed route/dates with fewer/more than 24 distinct hours:', len(partial_route_days))
display(partial_route_days.head(10))
print('Daily reported_hours distribution:')
display(daily.reported_hours.value_counts().sort_index().to_frame('route_date_rows'))

Hourly rows: 993600 | Full route/date/hour grid: 993600
Observed route/dates with fewer/more than 24 distinct hours: 0


,,reported_hours
bus_route,payment_date,


Daily reported_hours distribution:


,route_date_rows
reported_hours,
24,275896


## 15. Are there suspicious zero or low-demand periods to investigate?

Compare network daily totals to the same weekday one week earlier, using only
past dates. A ratio below 20% is an inspection flag, not a verified outage label.
Holidays, weather, disruptions, publisher processing and source coverage can all
change values. Exclusion decisions require source-specific evidence.

In [28]:
daily_totals = daily.groupby(daily_dates).published_boardings.sum().sort_index()
daily_totals = daily_totals.reindex(pd.date_range(daily_totals.index.min(), daily_totals.index.max()))
period_checks = daily_totals.to_frame('published_boardings')
period_checks['prior_same_weekday'] = daily_totals.shift(7)
period_checks['ratio_to_prior_week'] = (
    period_checks.published_boardings / period_checks.prior_same_weekday.where(
        period_checks.prior_same_weekday.gt(0)))
period_checks['all_zero'] = period_checks.published_boardings.eq(0)
period_checks['low_vs_prior_week'] = period_checks.ratio_to_prior_week.lt(0.20)
print('Entirely absent network dates:', int(period_checks.published_boardings.isna().sum()))
print('All-zero network dates:', int(period_checks.all_zero.sum()))
print('Low-total review dates:', int(period_checks.low_vs_prior_week.sum()))
display(period_checks.loc[period_checks.all_zero | period_checks.low_vs_prior_week].head(15))
zero_network_hours = hourly.groupby(hourly_times).ridership.sum()
print('All-zero network hours in hourly extract:', int(zero_network_hours.eq(0).sum()))
display(zero_network_hours.loc[zero_network_hours.eq(0)].head(10).to_frame('ridership'))

Entirely absent network dates: 0
All-zero network dates: 6
Low-total review dates: 6


,published_boardings,prior_same_weekday,ratio_to_prior_week,all_zero,low_vs_prior_week
2026-01-10,0,703470.0,0.0,True,True
2026-01-11,0,584023.0,0.0,True,True
2026-01-12,0,1193306.0,0.0,True,True
2026-01-13,0,1242679.0,0.0,True,True
2026-01-14,0,1256451.0,0.0,True,True
2026-01-15,0,1256354.0,0.0,True,True


All-zero network hours in hourly extract: 0


,ridership
transit_timestamp,


**Decision:** retain and flag suspicious source values. A complete grid of rows
can still contain publication placeholders or unobserved demand. No zero rows,
routes or dates are removed here, and no absence is filled with zero.

## 16. Do downloaded totals agree with separate publisher queries?

Sum the downloaded records by date and compare with separately requested
denominator files. This validates extraction arithmetic using a different query
of the same source, not an independent passenger measurement. Compare daily and
hourly extracts over their common window as a separate consistency check.

In [29]:
reconciliations = []
for name, frame, dates, paths, metrics in [
    ('daily', daily, daily_dates, sorted((RAW / 'history_daily').glob('*_denominator.csv')),
     ['published_boardings', 'published_transfers', 'source_rows']),
    ('hourly', hourly, hourly_times.dt.normalize(), sorted((RAW / 'ridership').glob('*_daily_denominator.csv')),
     ['ridership', 'transfers', 'source_rows']),
]:
    observed = frame[metrics].groupby(dates).sum().sort_index()
    references_for_dates = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    reference_dates = pd.to_datetime(references_for_dates.service_date).dt.normalize()
    expected = references_for_dates[metrics].set_index(reference_dates).sort_index()
    assert not expected.index.duplicated().any()
    assert observed.index.equals(expected.index), 'Missing denominator dates'
    differences = observed - expected
    reconciliations.append({'extract': name, 'dates': len(observed),
                            'dates_with_any_difference': int(differences.ne(0).any(axis=1).sum()),
                            'maximum_absolute_difference': differences.abs().max().max()})
display(pd.DataFrame(reconciliations))
hourly_daily_totals = hourly.groupby(hourly_times.dt.normalize()).ridership.sum()
common_dates = daily_totals.index.intersection(hourly_daily_totals.index)
common_difference = daily_totals.reindex(common_dates) - hourly_daily_totals.reindex(common_dates)
print('Common daily/hourly dates:', len(common_dates))
print('Common dates with different boarding totals:', int(common_difference.ne(0).sum()))

,extract,dates,dates_with_any_difference,maximum_absolute_difference
0,daily,638,0,0
1,hourly,92,0,0


Common daily/hourly dates: 92
Common dates with different boarding totals: 0


## 17. Do the route identifiers and date meanings support a join?

Inspect the actual feed rows instead of guessing a naming replacement. This
exposes SBS aliases directly. A code-level match is evidence for a crosswalk,
but is kept as a proposal until the mapping and route history are reviewed.

In [30]:
active_routes = routes.loc[routes.route_id.isin(trips.route_id),
                           ['route_id', 'route_short_name', 'route_long_name']].copy()
observed_codes = set(daily.bus_route)
active_routes['literal_short_name_match'] = active_routes.route_short_name.isin(observed_codes)
active_routes['route_id_alias_present'] = active_routes.route_id.isin(observed_codes)
unresolved_routes = active_routes.loc[~active_routes.literal_short_name_match]
print('Trip-referenced route records:', len(active_routes))
print('Literal short-name matches:', int(active_routes.literal_short_name_match.sum()))
print('Short-name aliases requiring review:', len(unresolved_routes))
display(unresolved_routes)
print('Dates: ridership uses payment clock dates; GTFS keeps overnight trips on their original service dates.')

Trip-referenced route records: 41
Literal short-name matches: 32
Short-name aliases requiring review: 9


,route_id,route_short_name,route_long_name,literal_short_name_match,route_id_alias_present
119,M14A+,M14A-SBS,Lower East Side - Abingdon Sq,False,True
120,M14D+,M14D-SBS,Lower East Side - Chelsea Piers,False,True
122,M15+,M15-SBS,East Harlem - South Ferry,False,True
126,M23+,M23-SBS,Chelsea Piers - East Side,False,True
128,M34+,M34-SBS,East Side - Javits Center,False,True
129,M34A+,M34A-SBS,Waterside - Port Authority Terminal,False,True
135,M60+,M60-SBS,West Side - LaGuardia Airport,False,True
138,M79+,M79-SBS,Yorkville - West Side,False,True
139,M86+,M86-SBS,Yorkville - West Side,False,True


Dates: ridership uses payment clock dates; GTFS keeps overnight trips on their original service dates.


## 18. What did the separate historical compatibility audit find?

Read the saved compatibility audit for all five historical archives. It selects
the latest whole snapshot captured on or before each service date, then applies
feed validity and calendars. This cell displays the saved results. Candidate
matches still require clock-date alignment for overnight service.

In [31]:
alignment_folder = ROOT / 'reports/historical_alignment'
alignment_summary = json.loads((alignment_folder / 'validation_summary.json').read_text())
current_hashes = dict(zip(source_inventory.file, source_inventory.sha256))
assert all(current_hashes[item['file']] == item['sha256'] for item in alignment_summary['sources'])
print('Saved alignment validation UTC:', alignment_summary['validated_utc'])
display(pd.Series({key: alignment_summary[key] for key in [
    'schedule_route_dates', 'literal_candidate_matches', 'candidate_routes', 'candidate_dates',
    'conservative_eligible_rows', 'conservative_eligible_routes', 'conservative_eligible_dates',
]}).to_frame('count'))
display(pd.Series(alignment_summary['compatibility_status_counts']).to_frame('route_date_rows'))
alignment_rows = pd.read_csv(alignment_folder / 'route_date_compatibility_audit.csv')
block_reviews = alignment_rows.loc[alignment_rows.block_overlap_review]
display(block_reviews[['service_date', 'bus_route', 'source_snapshot', 'status']])

Saved alignment validation UTC: 2026-10-29T13:41:41.204200+00:00


,count
schedule_route_dates,10588
literal_candidate_matches,8248
candidate_routes,32
candidate_dates,263
conservative_eligible_rows,1085
conservative_eligible_routes,7
conservative_eligible_dates,259


,route_date_rows
payment_date_service_date_boundary_review,7126
no_literal_ridership_match,2340
eligible_conservative_daily_comparison,1085
prior_date_context_missing,37


,service_date,bus_route,source_snapshot,status
5585,2026-05-24,M103,manhattan_20260409,payment_date_service_date_boundary_review
9575,2026-09-06,M103,manhattan_20260903,prior_date_context_missing


**Interpretation:** the smaller conservative subset reflects withholding policy,
not proof that all remaining data is incorrect. It favors routes/days without
overnight activity and is not representative of the whole network. Resolve SBS
mappings, midnight handling and flagged block overlaps before expanding comparisons.

## 19. Inspection findings and next decisions

This summary separates structural checks from issues requiring interpretation.
A flag is not a cleaning decision or an observed operating outcome.

In [32]:
findings = pd.DataFrame([
    ['Source integrity', f'{len(source_inventory)} matching receipts', 'PASS', 'Keep original bytes and provenance.'],
    ['Selected-feed duplicate keys', str(int(key_checks.duplicate_key_rows.sum())), 'CHECKED', 'Use correct keys; do not deduplicate display names.'],
    ['Selected-feed foreign references', str(int(references.unmatched_rows.sum())), 'CHECKED', 'Investigate any nonzero relationships before joins.'],
    ['Padded analysis fields', f'{int(field_checks.outer_spaces.sum())} text values', 'REVIEW', 'Inspect examples; explicitly parse numeric text and preserve source IDs.'],
    ['Shared route catalogue', f'{len(catalogue_only)} routes without templates', 'CONTEXT', 'Catalogue scope is wider than this borough timetable.'],
    ['Daily coverage', f'{int(coverage.absent_dates_inside_span.sum())} absent route/dates within observed spans', 'CHECKED', 'Full-window absences still need route-history context; do not zero-fill.'],
    ['Daily network zero values', f'{int(period_checks.all_zero.sum())} all-zero dates', 'REVIEW', 'Investigate source reporting; zeros alone do not prove no riders.'],
    ['SBS mapping', f'{len(unresolved_routes)} nonliteral short names', 'REVIEW', 'Validate explicit route-ID/short-name crosswalks.'],
    ['Midnight compatibility', str(alignment_summary['compatibility_status_counts'].get('payment_date_service_date_boundary_review', 0)), 'REVIEW', 'Reconstruct clock-date schedule activity before comparisons.'],
    ['Overlapping vehicle blocks', f'{len(block_reviews)} archived route/dates', 'REVIEW', 'Inspect source trip intervals before using service denominators.'],
    ['Passenger interpretation', 'Boarding estimates; no observed occupancy', 'LIMIT', 'Do not manufacture balanced/overcrowded labels.'],
], columns=['inspection_topic', 'finding', 'status', 'decision_or_next_check'])
display(findings)

,inspection_topic,finding,status,decision_or_next_check
0,Source integrity,57 matching receipts,PASS,Keep original bytes and provenance.
1,Selected-feed duplicate keys,0,CHECKED,Use correct keys; do not deduplicate display names.
2,Selected-feed foreign references,0,CHECKED,Investigate any nonzero relationships before joins.
3,Padded analysis fields,3648 text values,REVIEW,Inspect examples; explicitly parse numeric text and preserve source IDs.
4,Shared route catalogue,266 routes without templates,CONTEXT,Catalogue scope is wider than this borough timetable.
5,Daily coverage,0 absent route/dates within observed spans,CHECKED,Full-window absences still need route-history context; do not zero-fill.
6,Daily network zero values,6 all-zero dates,REVIEW,Investigate source reporting; zeros alone do not prove no riders.
7,SBS mapping,9 nonliteral short names,REVIEW,Validate explicit route-ID/short-name crosswalks.
8,Midnight compatibility,7126,REVIEW,Reconstruct clock-date schedule activity before comparisons.
9,Overlapping vehicle blocks,2 archived route/dates,REVIEW,Inspect source trip intervals before using service denominators.


## 20. What this inspection supports

This inspection documents source structure, repeatable checks and limitations
that require review before cleaning. These checks do not establish that every
route/date pair is suitable for modelling or measure actual delays, occupancy,
operated trips or optimization benefits. Fingerprints, duplicate checks and successful parsing do not prove that
an agency estimate reflects every real passenger or schedule change.

Remaining inspection/compatibility work includes verifying anomalous periods,
route histories and SBS aliases, handling overnight clock dates, reviewing block
conflicts, and obtaining more archives if the intended study scope needs them.
Cleaning decisions and engineered model features belong to the next stage.

Reusable implementations: `scripts/week1_inspection.py` and
`scripts/validate_historical_alignment.py`. The original fare-category inspection
is limited to a 10,000-row sample; aggregated extract checks do not audit every
original payment-category record.

In [33]:
# Confirm that inspection did not change any original source file.
for row in source_inventory.itertuples(index=False):
    with (ROOT / row.file).open('rb') as handle:
        assert hashlib.file_digest(handle, 'sha256').hexdigest() == row.sha256
print('Inspection finished. All original source fingerprints remain unchanged.')

Inspection finished. All original source fingerprints remain unchanged.


**Continue to notebook 02:** [Data cleaning](02_data_cleaning.ipynb) records the subsequent decisions. [Feature engineering](03_feature_engineering.ipynb) extends the historical audit with exact route-ID mapping, calendar completeness review and clock-date integration. Inspection results above remain the evidence at this stage; the later notebooks describe what was resolved and what remains uncertain.